# NBA 3.1 - Combine target variable and create model splits

Joins the HCP x cohort target (`KERENDIA_NBRX` from `Target_Variable/HCP_Target_Variable`) onto the combined physician data (`NBA_3_1_Combine_Physician_Data`) as **`TARGET_NBRX`**, on `HCP_COHORT_ID`, and writes three Delta splits:

| Split | Cohorts |
| --- | --- |
| Training | c1 - c24 |
| Validation | c25 - c26 |
| Backtesting (Holdout) | c27 - c28 |

A split is written only when every one of its cohorts is in the combined table. Combined rows with no target row (HCPs outside the target universe) are dropped and saved to a side folder.

In [0]:
from pyspark.sql import functions as F
import os

dbutils.widgets.text("RUN_NAME", "combined_2026_11_24")
RUN_NAME = dbutils.widgets.get("RUN_NAME").strip()
assert RUN_NAME, "RUN_NAME is required"

VOL31 = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/"
DEV   = VOL31 + "hcp_level_model/dev_pipeline/notebook_output/"

COMBINED_PATH = f"{DEV}{RUN_NAME}/hcp_combined/kerendia_nba_3_1_hcp_selected_features_data"
TARGET_PATH   = VOL31 + "notebook_output/target_variable/target_variable"

SPLIT_DIR = f"{DEV}{RUN_NAME}/model_data/"
SPLITS = {   # split name -> (first cohort, last cohort, output path)
    "training":    (1, 24,  SPLIT_DIR + "kerendia_nba_3_1_hcp_training_data"),
    "validation":  (25, 26, SPLIT_DIR + "kerendia_nba_3_1_hcp_validation_data"),
    "backtesting": (27, 28, SPLIT_DIR + "kerendia_nba_3_1_hcp_backtesting_holdout_data"),
}
MISSING_TARGET_PATH = SPLIT_DIR + "rows_without_target"
TARGET_NAME = "TARGET_NBRX"

for _p in [p for *_, p in SPLITS.values()] + [MISSING_TARGET_PATH]:
    assert _p.startswith(VOL31) and "kerendia_nba_3_volume" not in _p, f"refusing to write outside the 3.1 volume: {_p}"
print("Combined:", COMBINED_PATH)
print("Target:  ", TARGET_PATH)
print("Splits:  ", SPLIT_DIR)

In [0]:
target = (spark.read.format("delta").load(TARGET_PATH)
          .select("HCP_COHORT_ID", F.col("KERENDIA_NBRX").cast("int").alias(TARGET_NAME)))
n_t = target.count()
assert target.select("HCP_COHORT_ID").distinct().count() == n_t, "target HCP_COHORT_ID not unique"
assert target.filter(F.col(TARGET_NAME).isNull()).count() == 0
print(f"Target rows: {n_t:,}")

In [0]:
combined = (spark.read.format("delta").load(COMBINED_PATH)
            .withColumn("_COHORT_N", F.expr("int(substr(COHORT_ID, 2))")))
assert TARGET_NAME not in combined.columns, f"{TARGET_NAME} already in the combined table"

joined = combined.join(target, "HCP_COHORT_ID", "left")

# Rows per cohort: combined rows, rows with a target, rows without one
coverage = (joined.groupBy("COHORT_ID", "_COHORT_N", "COHORT_SET")
            .agg(F.count("*").alias("rows"),
                 F.sum(F.col(TARGET_NAME).isNotNull().cast("int")).alias("with_target"),
                 F.sum(F.col(TARGET_NAME).isNull().cast("int")).alias("without_target"),
                 F.sum(TARGET_NAME).alias("total_nbrx"))
            .orderBy("_COHORT_N"))
cov_pd = coverage.toPandas()
display(cov_pd)
present = set(cov_pd["_COHORT_N"].tolist())
print(f"Cohorts in the combined table: {sorted(present)}")
print(f"Rows without a target: {int(cov_pd.without_target.sum()):,}")

In [0]:
# HCPs in the combined universe but not in the target universe: kept aside, not imputed
missing = joined.filter(F.col(TARGET_NAME).isNull()).select("HCP_COHORT_ID", "PROVIDER_ID", "COHORT_ID", "COHORT_SET")
n_missing = missing.count()
if n_missing:
    missing.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(MISSING_TARGET_PATH)
    print(f"Saved {n_missing:,} rows ({missing.select('PROVIDER_ID').distinct().count():,} HCPs) -> {MISSING_TARGET_PATH}")
else:
    print("Every combined row has a target")

In [0]:
model_data = joined.filter(F.col(TARGET_NAME).isNotNull())
split_report = []
for name, (lo, hi, path) in SPLITS.items():
    need = set(range(lo, hi + 1))
    if not need <= present:
        print(f"{name}: SKIPPED, cohorts not in the combined table yet: {sorted(need - present)}")
        split_report.append((name, f"c{lo}-c{hi}", "skipped", 0, path))
        continue
    df = (model_data.filter(F.col("_COHORT_N").between(lo, hi))
          .withColumn("SPLIT", F.lit(name)).drop("_COHORT_N"))
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(path)
    n = spark.read.format("delta").load(path).count()
    print(f"{name}: c{lo}-c{hi} -> {n:,} rows -> {path}")
    split_report.append((name, f"c{lo}-c{hi}", "written", n, path))

display(spark.createDataFrame(split_report, "split string, cohorts string, status string, rows long, path string"))

In [0]:
# Per written split: rows = combined rows with a target, unique keys, only its cohorts, no null target,
# and TARGET_NBRX equal to KERENDIA_NBRX in the target table
tv_full = spark.read.format("delta").load(TARGET_PATH).select("HCP_COHORT_ID", "KERENDIA_NBRX")
seen_keys = None
for name, (lo, hi, path) in SPLITS.items():
    if not set(range(lo, hi + 1)) <= present:
        continue
    s = spark.read.format("delta").load(path)
    n = s.count()
    exp = int(cov_pd[(cov_pd._COHORT_N >= lo) & (cov_pd._COHORT_N <= hi)].with_target.sum())
    assert n == exp, (name, n, exp)
    assert s.select("HCP_COHORT_ID").distinct().count() == n, f"{name}: duplicate keys"
    bad = s.filter(~F.expr("int(substr(COHORT_ID, 2))").between(lo, hi)).count()
    assert bad == 0, f"{name}: {bad} rows from other cohorts"
    assert s.filter(F.col(TARGET_NAME).isNull()).count() == 0
    mism = (s.select("HCP_COHORT_ID", TARGET_NAME).join(tv_full, "HCP_COHORT_ID")
             .filter(F.col(TARGET_NAME) != F.col("KERENDIA_NBRX")).count())
    assert mism == 0, f"{name}: {mism} target mismatches"
    keys = s.select("HCP_COHORT_ID")
    if seen_keys is not None:
        assert keys.join(seen_keys, "HCP_COHORT_ID", "left_semi").count() == 0, f"{name}: overlaps another split"
    seen_keys = keys if seen_keys is None else seen_keys.unionByName(keys)
    print(f"{name}: {n:,} rows, {len(s.columns):,} columns, TARGET_NBRX sum {s.agg(F.sum(TARGET_NAME)).first()[0]:,} - checks passed")